# Flux-parameter sensitivity of several geometries

Evaluates the forward-folding analysis loss on a fixed event sample for each geometry (no gradients) and compares how well each one constrains the signal flux parameter $\theta$.

In the Asimov (Gaussian) approximation the log-likelihood around the true values $\hat\theta$ is quadratic in the Fisher matrix $F$:
$$-2\,\Delta\ln L(\theta) = (\theta-\hat\theta)^\top F\,(\theta-\hat\theta).$$
Scanning one parameter $\theta_i$ with all others **fixed** at their true values gives $-2\Delta\ln L = F_{ii}(\theta_i-\hat\theta_i)^2$ (equivalently $-\Delta\ln L = \tfrac12 F_{ii}(\theta_i-\hat\theta_i)^2$; the formula with both the 2 and the ½ mixes these two forms). If the other (nuisance) parameters are **re-fitted** at every point of the scan (profiling), the curvature is the Schur complement of the nuisance block,
$$-2\,\Delta\ln L_\text{prof} = \big(F_{ii} - F_{i\nu}F_{\nu\nu}^{-1}F_{\nu i}\big)(\theta_i-\hat\theta_i)^2 = \frac{(\theta_i-\hat\theta_i)^2}{(F^{-1})_{ii}},$$
which is never narrower than the fixed-nuisance parabola. Where a parabola crosses $-2\Delta\ln L = 1$ is the 1σ (68.3%) interval, $3.84$ the 95% one. The profiled $\sigma$ is what the analysis loss minimises.

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '3'
import pickle
import time
import torch
import nugget
from nugget.utils import jax_bridge
from nugget.samplers.sim_weights import event_arrays, generation_params, get_simulation_weights
from nugget.samplers.flux_weights import get_flux_weights
from nugget.utils.grad_accumulation import disjoint_batches
from nugget.losses.analysis_loss import fisher_constraints

In [ ]:
device = 'cuda:0'
jax_bridge.configure_jax(device=device) # before anything imports JAX (pyForwardFolding, for the flux weights)

In [ ]:
signal_sampler = nugget.samplers.range_sampler.RangeSampler(
    device=device,
    E_min=1e2,
    E_max=1e6,
    energy_dist='log_uniform',
    injection_radius=1100,
    chunk_size=10000,
    seed=1,  # same events every time the notebook runs
)

In [ ]:
num_events_total = int(1e5)
signal_events = signal_sampler.sample_events(num_events_total)

In [ ]:
live_time = 10 * 365.25 * 24 * 3600  # 10 years in seconds
friend_config = '../other/friend_config.yaml'
pyff_config = '../other/pyff_config.yaml'

# pass filename=... to both if nugget/assets/data is not staged on this machine
xsec = nugget.losses.effective_area.CrossSection()
transmission = nugget.losses.effective_area.TransmissionProb()

ev = event_arrays(signal_events)  # arrival convention, as the transmission and atmospheric fluxes expect
sim_weights = get_simulation_weights(
    ev['energy'], ev['cos_zenith'],
    xsec=xsec, transmission=transmission, batched=True, column_depth=ev['column_depth'],
    **generation_params(signal_sampler),
) # GeV cm^2 sr, normalised to N_gen = num_events_total
flux, grad_flux = get_flux_weights(
    ev['energy'], ev['zenith'], ev['azimuth'],
    friend_config, pyff_config, device=device,
) # GeV^-1 cm^-2 s^-1 sr^-1, and its gradient w.r.t. each flux parameter
print('flux parameters:', list(grad_flux))
print(f'expected events before trigger in {live_time / 3.15576e7:.0f} yr: {(live_time * sim_weights * flux).sum().item():.4g}')

In [ ]:
# Same geometry class and settings as the optimization, so the saved geom_dicts re-initialise correctly.
geometry = nugget.geometries.DynamicString.DynamicString(
                device=device,
                hex_type='hexagonal',
                domain_size=500,
                dim=3,
                n_strings=70,
                points_per_string=20,
                custom_z_spacing=40,
            )

visualizer = nugget.utils.vis_tools.Visualizer(
    device=device,
    dim=3,
    domain_size=1400,
)

# Wraps the losses like HybridOptimizer: the analysis loss pools its histograms over all batches before the Fisher head.
evaluator = nugget.utils.hybrid_evaluator.HybridEvaluator(
    device=device,
    geometry=geometry,
    visualizer=visualizer,
)

In [ ]:
HIT_CKPT   = './flow_models/best_mc_hit_model_v1r.pt'
LY_CKPT    = './flow_models/best_mc_ly_muon_flow_model_v1r.pt'
ATIME_CKPT = './flow_models/best_mc_atime_muon_flow_model_v1r.pt'
nusmoothie = nugget.surrogates.NuSmoothie.NuSmoothie(
    device=device,
    domain_size=10000,          # overwritten per model by load_model
    hit_checkpoint=HIT_CKPT,
    ly_checkpoint=LY_CKPT,
    atime_checkpoint=ATIME_CKPT,
    n_steps=8,
    ly_n_samples=1,
    batch_size=100000,
    float32=True,
)

# One per-OM Fisher network gives both the energy and the zenith resolution.
om_fisher = nugget.losses.fisher_info_om_net.OMFisherNetLoss(
    device=geometry.device,
).load_model('./fisher_models/best_om_muon_fisher_net_v3mc.pt')

trigger_loss = nugget.losses.photon_trigger.PhotonCountTrigger(
    device=geometry.device,
    n_photons=10,
    softness=0.2,
    cut='sigmoid',
    count='photons',
    ly_prune_below=1e-3,
)

analysis_loss = nugget.losses.analysis_loss.AnalysisLoss(
    device=geometry.device,
    trigger_loss=trigger_loss,
    fisher_loss=om_fisher,
    xsec=xsec,
    transmission=transmission,
)

loss_func_dict = {
    'fisher_analysis_loss': analysis_loss,
}

In [ ]:
batch_size = 1000
n_eval_batches = int(1e5 / batch_size)  # events used per geometry = n_eval_batches * batch_size (at most num_events_total)

# Disjoint batches of the precomputed events. The sim weights are normalised to all num_events_total
# events, so each batch scales them up to stand for the full sample and the batches are averaged ('mean').
batch_hook = disjoint_batches(
    {
        'signal_event_params': signal_events,
        'analysis_sim_weights': sim_weights,
        'analysis_flux_weights': flux,
        'analysis_grad_flux_weights': grad_flux,
    },
    batch_size=batch_size,
    rescale_keys=('analysis_sim_weights',),
)

loss_params = {
    # analysis loss
    'signal_sampler': signal_sampler,
    'num_events': batch_size,
    'live_time': live_time,
    'friend_config': friend_config,
    'pyff_config': pyff_config,  # also gives the true parameter values the scan is centred on
    'analysis_binning_var_name': ['energy', 'zenith'],
    'analysis_num_bins': 30,
    'analysis_signal_flux_var_names': ['astro_norm'],
    'analysis_optimality': 'a',
    'use_relative_energy': False,
    'fisher_info_chunk_size': 100000,
    'analysis_energy_res_selection': True,
    # 'analysis_resample_vertices': True,   # vertices on a disk fitted around each geometry (then sim weights are per batch)

    # trigger
    'signal_surrogate_func': nusmoothie.light_yield_surrogate,
    'use_batched_surrogate': True,
    'binned_trigger_batch_size': 10000,
    'analysis_perfect_efficiency': False,

    # accumulation over batches, exactly as during optimization
    'loss_repeats':      {'fisher_analysis_loss': n_eval_batches},
    'loss_accumulators': {'fisher_analysis_loss': 'analysis'},
    'on_repeat':         {'fisher_analysis_loss': batch_hook},
    'loss_reduction':    {'fisher_analysis_loss': 'mean'},
    'empty_cache_per_repeat': True,
}

In [ ]:
# Geometries to compare: a name and a saved geom_dict (None = the geometry's own default layout).
geom_files = {
    'initial hexagonal': None,
    'optimised': './best_analysis_geom_2.pkl',
}

geom_dicts = {}
for name, path in geom_files.items():
    if path is None:
        geom_dicts[name] = geometry.initialize_points()
    else:
        with open(path, 'rb') as f:
            geom_dicts[name] = pickle.load(f)
print(list(geom_dicts))

In [ ]:
plot_types = [
    'flux_likelihood_scan',      # -2 Delta ln L parabolas, one per geometry (overlaid)
    'string_xy',                 # the layouts, one panel per geometry
    'angular_resolution_vs_energy',
    'energy_resolution_vs_energy',
]

vis_kwargs = {
    'plot_types': plot_types,
    'likelihood_scan_param': 'astro_norm',
    'likelihood_scan_nsigma': 3,
    'likelihood_scan_conditional': True,  # dashed: the other flux parameters fixed
    'resolution_logy_energy': True,
    'resolution_logy_angular': True,
    'use_relative_energy': True,
    'resolution_stat': 'median',
}

t0 = time.time()
result = evaluator.evaluate_multi(
    geom_dicts=geom_dicts,
    loss_func_dict=loss_func_dict,
    loss_params_dict=loss_params,
    vis_kwargs=vis_kwargs,
    visualize=True,
    seed=0,                 # same event batches for every geometry
    clear_cuda_cache=True,
)
print(f'evaluation took {time.time() - t0:.1f} s')

In [ ]:
# 1-sigma constraint on each signal flux parameter, with the other parameters fixed and profiled.
reference = None
print(f"{'geometry':<22}{'param':<14}{'true':>8}{'sigma (fixed)':>16}{'sigma (profiled)':>18}{'vs first':>10}")
for name, res in result.items():
    vk = res['vis_kwargs']
    cons = fisher_constraints(vk['analysis_fisher_matrix'], vk['analysis_param_names'], vk['flux_param_names'])
    values = vk.get('analysis_param_values', {})
    for param, (sigma_fixed, sigma_prof) in cons.items():
        reference = reference or {}
        reference.setdefault(param, sigma_prof)
        print(f"{name:<22}{param:<14}{values.get(param, float('nan')):>8.3g}{sigma_fixed:>16.4g}{sigma_prof:>18.4g}"
              f"{sigma_prof / reference[param]:>10.3f}")